# DLL Lab — 12 February 2026
## CNN — Image Classification on CIFAR-10
**Done By:** Student | **Course:** AI-612


In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tensorflow import keras
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Flatten, Dropout, BatchNormalization
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import confusion_matrix
import warnings; warnings.filterwarnings('ignore')
print('Libraries loaded.')


Libraries loaded.


## Load CIFAR-10

In [2]:
(X_train, y_train), (X_test, y_test) = cifar10.load_data()
X_train = X_train.astype('float32') / 255.0
X_test  = X_test.astype('float32')  / 255.0
y_train_oh = to_categorical(y_train, 10)
y_test_oh  = to_categorical(y_test, 10)
class_names = ['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck']
print(f'Train: {X_train.shape}  Test: {X_test.shape}')


Train: (50000, 32, 32, 3)  Test: (10000, 32, 32, 3)


In [3]:
fig, axes = plt.subplots(2, 5, figsize=(11, 4))
for i, ax in enumerate(axes.flatten()):
    idx = np.where(y_train.flatten() == i)[0][0]
    ax.imshow(X_train[idx])
    ax.set_title(class_names[i]); ax.axis('off')
plt.suptitle('CIFAR-10 Sample Images', fontsize=12)
plt.tight_layout(); plt.show()


## Build CNN

In [4]:
model = Sequential([
    Conv2D(32, (3,3), activation='relu', padding='same', input_shape=(32,32,3)),
    BatchNormalization(),
    MaxPooling2D(),
    Conv2D(64, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(),
    Conv2D(128, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(),
    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.4),
    Dense(10, activation='softmax')
])
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()


Model: "sequential"
_________________________________________________________________
 Layer (type)           Output Shape              Param #   
 conv2d (Conv2D)        (None, 32, 32, 32)        896       
 batch_normalization    (None, 32, 32, 32)        128       
 max_pooling2d          (None, 16, 16, 32)        0         
 conv2d_1 (Conv2D)      (None, 16, 16, 64)        18,496    
 batch_normalization_1  (None, 16, 16, 64)        256       
 max_pooling2d_1        (None, 8, 8, 64)          0         
 conv2d_2 (Conv2D)      (None, 8, 8, 128)         73,856    
 batch_normalization_2  (None, 8, 8, 128)         512       
 max_pooling2d_2        (None, 4, 4, 128)         0         
 flatten (Flatten)      (None, 2048)              0         
 dense (Dense)          (None, 256)               524,544   
 dropout (Dropout)      (None, 256)               0         
 dense_1 (Dense)        (None, 10)                2,570     
Total params: 621,258
______________________________________

## Train

In [5]:
history = model.fit(X_train, y_train_oh, epochs=20, batch_size=64,
                    validation_data=(X_test, y_test_oh), verbose=1)


Epoch  1/20 — loss: 1.4812 — accuracy: 0.4723 — val_accuracy: 0.5312
Epoch  5/20 — loss: 0.8934 — accuracy: 0.6845 — val_accuracy: 0.6978
Epoch 10/20 — loss: 0.6523 — accuracy: 0.7712 — val_accuracy: 0.7534
Epoch 15/20 — loss: 0.4978 — accuracy: 0.8234 — val_accuracy: 0.7723
Epoch 20/20 — loss: 0.3880 — accuracy: 0.8634 — val_accuracy: 0.7848


In [6]:
_, test_acc = model.evaluate(X_test, y_test_oh, verbose=0)
print(f'Test Accuracy: {test_acc:.4f}')

y_pred = np.argmax(model.predict(X_test), axis=1)
cm = confusion_matrix(y_test.flatten(), y_pred)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(history.history['accuracy'], label='Train')
axes[0].plot(history.history['val_accuracy'], label='Val')
axes[0].set_title('Accuracy over Epochs')
axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(True, alpha=0.3)
sns.heatmap(cm, annot=True, fmt='d', cmap='YlOrRd', ax=axes[1],
            xticklabels=class_names, yticklabels=class_names)
axes[1].set_title('Confusion Matrix')
axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout(); plt.show()


Test Accuracy: 0.7848


## Summary
| | Value |
|---|---|
| Dataset | CIFAR-10 (50k train / 10k test) |
| Architecture | 3× Conv-BN-Pool → Dense |
| Test Accuracy | **78.48%** |
| Epochs | 20 |
